In [ ]:
import math
from pypdf import PdfReader, PdfWriter, PageObject, Transformation

# ==============================================================================
# CONFIGURACIÓN (Ajusta tus variables aquí)
# ==============================================================================
pdf_filename = "0003_logica.pdf"  # Debe estar en la misma carpeta

hojas_por_booklet = 5  # Cantidad de hojas físicas plegadas (5 hojas = 20 páginas)
tamano_hoja_final = "A4"  # Opciones: "A4" o "LEGAL"

carillas_blancas_inicio = 2  # Carillas en blanco al principio del libro (0, 1 o 2)

# Factor de escala manual: 1.0 es el tamaño original. 
# Usa valores mayores a 1.0 para agrandar (ej: 1.15 es +15%) 
# o menores a 1.0 para achicar (ej: 0.85 es -15%).
escala_manual_porcentaje = 0.69

# Flag de Resaltados: 
# False -> Elimina todos los textos resaltados en amarillo y marcas del PDF original.
# True  -> Mantiene los resaltados tal como están en el archivo original.
keep_resaltado = False  

# Reemplaza ".pdf" inyectando las variables de configuración en el nombre
sufijo_dinamico = f"_bookletSize{hojas_por_booklet}_{tamano_hoja_final}.pdf"
output_filename = pdf_filename.replace(".pdf", sufijo_dinamico)


# ==============================================================================
# CONSTANTES DE MEDIDAS (En puntos PostScript: 1 pulgada = 72 puntos)
# ==============================================================================
# Definimos el tamaño de la hoja física final en HORIZONTAL (Apaisada)
MEDIDAS_HOJA = {
    "A4": {"ancho": 841.89, "alto": 595.28},     # A4 Horizontal
    "LEGAL": {"ancho": 1008.0, "alto": 612.0}    # Legal/Oficio Horizontal
}

# ==============================================================================
# FUNCIONES MODULARES
# ==============================================================================
def generar_patron_booklet(paginas_totales_booklet):
    """
    Genera el orden de impresión tradicional para un booklet de N páginas.
    Ejemplo para 8 páginas: [8, 1, 2, 7, 6, 3, 4, 5] (1-indexado de forma lógica)
    """
    orden = []
    izq = 1
    der = paginas_totales_booklet
    while izq < der:
        if izq % 2 != 0:
            orden.extend([der, izq])
        else:
            orden.extend([izq, der])
        izq += 1
        der -= 1
    return orden

def redimensionar_y_centrar(pagina_origen, ancho_destino, alto_destino, factor_escala=1.0):
    """
    Aplica una escala manual basada en porcentaje a la página original
    y la centra de manera perfecta en la mitad de la hoja final de impresión.
    """
    # Eliminar anotaciones/resaltados si el flag está en False
    if not keep_resaltado and "/Annots" in pagina_origen:
        del pagina_origen["/Annots"]

    # Usar cropbox si está definido, si no recurrir a mediabox
    caja = pagina_origen.cropbox if pagina_origen.cropbox else pagina_origen.mediabox
    ancho_orig = float(caja.width)
    alto_orig = float(caja.height)
    
    # Calcular el tamaño que tendrá la página una vez escalada con tu porcentaje
    ancho_escalado = ancho_orig * factor_escala
    alto_escalado = alto_orig * factor_escala
    
    # Calcular el espacio necesario para que quede perfectamente centrada
    # dentro de su mitad correspondiente de la hoja física
    margen_x = (ancho_destino - ancho_escalado) / 2
    margen_y = (alto_destino - alto_escalado) / 2
    
    # Ajustar las coordenadas de traslación tomando en cuenta el origen del PDF (left/bottom)
    tx = margen_x - (float(caja.left) * factor_escala)
    ty = margen_y - (float(caja.bottom) * factor_escala)
    
    # Crear la matriz de transformación con la escala y el centrado correctos
    transformacion = Transformation().scale(factor_escala, factor_escala).translate(tx, ty)
    
    # Crear una página contenedora vacía con el tamaño exacto de la mitad de la hoja final
    pag_mitad = PageObject.create_blank_page(width=ancho_destino, height=alto_destino)
    pag_mitad.merge_page(pagina_origen)
    pag_mitad.add_transformation(transformacion)
    
    return pag_mitad

# ==============================================================================
# PROCESAMIENTO PRINCIPAL
# ==============================================================================
# 1. Cargar el PDF original
reader = PdfReader(pdf_filename)
paginas_originales = list(reader.pages)

# 2. Aplicar carillas en blanco al inicio si se requiere
if carillas_blancas_inicio > 0:
    # Creamos páginas en blanco idénticas en tamaño a la primera página del PDF
    ancho_ref = paginas_originales[0].mediabox.width
    alto_ref = paginas_originales[0].mediabox.height
    for _ in range(carillas_blancas_inicio):
        pag_blanca = PageObject.create_blank_page(width=ancho_ref, height=alto_ref)
        paginas_originales.insert(0, pag_blanca)

total_paginas_con_blancas = len(paginas_originales)
print(f"Páginas totales a procesar (incluyendo blancas iniciales): {total_paginas_con_blancas}")

# 3. Calcular la estructura de los booklets
paginas_por_booklet = hojas_por_booklet * 4  # Cada hoja física plegada alberga 4 carillas/páginas
total_booklets = math.ceil(total_paginas_con_blancas / paginas_por_booklet)
patron_base = generar_patron_booklet(paginas_por_booklet)

# Medidas de la hoja final
medidas_finales = MEDIDAS_HOJA[tamano_hoja_final.upper()]
ancho_hoja_final = medidas_finales["ancho"]
alto_hoja_final = medidas_finales["alto"]
ancho_mitad_hoja = ancho_hoja_final / 2  # El espacio que le corresponde a cada página del PDF

writer = PdfWriter()

# 4. Iterar sobre cada booklet e imponer las páginas
for b in range(total_booklets):
    offset_actual = b * paginas_por_booklet
    
    # El patrón base nos da pares de páginas. Recorremos de a 2 (Izquierda y Derecha de la hoja física)
    for i in range(0, len(patron_base), 2):
        # Obtener los números lógicos del patrón base
        num_pag_izq = patron_base[i]
        num_pag_der = patron_base[i+1]
        
        # Mapear a los índices reales indexados en 0
        idx_real_izq = offset_actual + (num_pag_izq - 1)
        idx_real_der = offset_actual + (num_pag_der - 1)
        
        # Obtener o generar página izquierda
        if idx_real_izq < total_paginas_con_blancas:
            pag_izq_orig = paginas_originales[idx_real_izq]
        else:
            # Si excede el libro, se genera una página vacía de relleno al final
            pag_izq_orig = PageObject.create_blank_page(width=paginas_originales[0].mediabox.width, height=paginas_originales[0].mediabox.height)
            
        # Obtener o generar página derecha
        if idx_real_der < total_paginas_con_blancas:
            pag_der_orig = paginas_originales[idx_real_der]
        else:
            # Relleno al final
            pag_der_orig = PageObject.create_blank_page(width=paginas_originales[0].mediabox.width, height=paginas_originales[0].mediabox.height)
            
        # Redimensionar y adaptar ambas páginas a la mitad correspondiente usando la escala manual
        mitad_izq = redimensionar_y_centrar(pag_izq_orig, ancho_mitad_hoja, alto_hoja_final, escala_manual_porcentaje)
        mitad_der = redimensionar_y_centrar(pag_der_orig, ancho_mitad_hoja, alto_hoja_final, escala_manual_porcentaje)
        
        # Crear la hoja física final (A4 o Legal Horizontal) vacía
        hoja_fisica = PageObject.create_blank_page(width=ancho_hoja_final, height=alto_hoja_final)
        
        # Fusionar la mitad izquierda (en la coordenada X = 0)
        hoja_fisica.merge_page(mitad_izq)
        
        # Fusionar la mitad derecha (desplazada en X la mitad del ancho total)
        hoja_fisica.merge_translated_page(mitad_der, tx=ancho_mitad_hoja, ty=0)
        
        # Añadir al documento de salida
        writer.add_page(hoja_fisica)

# 5. Guardar el archivo definitivo
with open(output_filename, "wb") as f_out:
    writer.write(f_out)

print(f"\n¡Proceso completado con éxito!")
print(f"Estructura: {total_booklets} booklets de {hojas_por_booklet} hojas cada uno.")
print(f"Archivo generado: '{output_filename}' en tamaño {tamano_hoja_final} Horizontal.")


Páginas totales a procesar (incluyendo blancas iniciales): 44

¡Proceso completado con éxito!
Estructura: 3 booklets de 5 hojas cada uno.
Archivo generado: '0003_logica_bookletSize5_A4.pdf' en tamaño A4 Horizontal.
